# Lekcja 07 - Wzorzec projektowy planowania

Ten notatnik demonstruje **Wzorzec projektowy planowania** dla agentów AI przy użyciu Microsoft Agent Framework.  
Nauczysz się, jak rozbić złożone zapytania podróżnicze na ustrukturyzowane podzadania, przydzielić je specjalistycznym agentom  
i wykonać powstały plan — wszystko to przy użyciu ustrukturyzowanego wyniku opartego na modelach Pydantic.


## Instalacja


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Rozkład zadań

Rozkład zadań jest rdzeniem wzorca projektowego planowania. Zamiast prosić jednego agenta o obsługę złożonego zadania od początku do końca, dzielimy problem na mniejsze, dobrze zdefiniowane **podzadania**. Każde podzadanie jest przydzielone specjalistycznemu agentowi (np. loty, hotele, atrakcje) z jasno określonymi priorytetami i kolejnością zależności.

Takie podejście daje kilka korzyści:
- **Przejrzystość**: każde podzadanie ma pojedynczą odpowiedzialność.
- **Równoległość**: niezależne podzadania mogą być wykonywane równocześnie.
- **Niezawodność**: błędy są izolowane do poszczególnych podzadań.
- **Śledzenie budżetu**: koszty są szacowane dla każdego podzadania i sumowane.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Tworzenie agenta planującego ze strukturą wyjściową

Agent planujący działa jako **koordynator recepcji**. Na podstawie ogólnego zapytania dotyczącego podróży
generuje ustrukturyzowany `TravelPlan` — rozkładając zapytanie na podzadania, ustalając priorytety
i identyfikując zależności, tak aby konsjerż lub warstwa wykonawcza mogły zrealizować zadanie.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Wykonywanie Planu za pomocą Specjalistycznych Narzędzi

Gdy agent recepcji sporządził ustrukturyzowany plan, wykonuje go **agent concierge**.
Każde narzędzie specjalistyczne obsługuje jedną kategorię podzadań (loty, hotele, atrakcje). Concierge
przechodzi kolejno przez podzadania planu w kolejności zależności i przekazuje każde z nich
do odpowiedniego narzędzia.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Podsumowanie

W tej lekcji nauczyłeś się **wzoru projektowego planowania** dla agentów AI:

1. **Dekonstrukcja zadania** — Agent planujący na recepcji rozbija złożone zapytanie podróżne na
   ustrukturyzowane podzadania za pomocą modeli Pydantic, przydzielając każde z nich specjaliście wraz z priorytetami
   i zależnościami.
2. **Ustrukturyzowany wynik** — Przekazując `response_format`, agent zwraca zwalidowany
   obiekt `TravelPlan` zamiast dowolnego tekstu, co zapewnia niezawodne przetwarzanie dalsze.
3. **Wykonanie planu** — Agent konsjerża iteruje przez podzadania, używając specjalistycznych narzędzi
   (`book_flight`, `reserve_hotel`, `book_activity`), aby zrealizować plan i raportować wyniki.

Ten wzór oddziela *co zrobić* (planowanie) od *jak to zrobić* (wykonanie), czyniąc agentów
bardziej modułowymi, testowalnymi i łatwiejszymi do rozszerzania.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:  
Dokument ten został przetłumaczony przy użyciu usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Chociaż dokładamy starań, aby tłumaczenie było jak najbardziej precyzyjne, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym powinien być traktowany jako źródło wiążące. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
